# 02-短期记忆
## 1、基于内存的持久化器
### 1.1举例1：没有记忆

In [1]:
import os
from typing import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent = create_agent(
model=model,
tools=[]
)
print("\n第一轮对话：")
response1 = agent.invoke({
"messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
"messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")

### 1.2举例2拥有记忆

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()    #具体实例，创建了记忆存储
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer
)
#同一个thread_id共享记忆
config = {
    "configurable":{
        "thread_id":1

    }
}
print("\n第一轮对话：")
response1 = agent.invoke({
"messages": [HumanMessage("我叫张三")] },
        config = config )
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
        "messages": [HumanMessage("我叫什么？")]},
        config = config )
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 你刚才告诉我，你叫**张三**。😊

有什么需要帮忙的吗？


In [ ]:
from rich import print as rprint

thread_1_state =agent.get_state(config)
rprint(thread_1_state)

In [3]:
print("\n第三轮对话：")
response3 = agent.invoke({
        "messages": [HumanMessage("我刚才问了什么问题")]},
        config = config )
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：**“我叫什么？”**

而在这之前，你第一次说的是：**“我叫张三”**。😊
